# Stage 7 — GSE270828 processed single-cell metadata audit (Google Colab)

**Scientific goal:** inspect a processed `.h5ad` file for HAR / sgRNA / batch / non-targeting control coverage, **without** calculating spurious differential-expression P values.

**Preferred input:** original experiment GSE270828, but processed PRISM AnnData may be used for a *feasibility pilot only*. The PRISM object is described as **87,896 cells × 19,698 features**, a filtered subset, whereas the original authors profiled more than **188,000 NSCs**. It must **not** be called the complete original screen.

**Source links:**
- Original GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE270828
- PRISM dataset listing / download folder: https://github.com/Peekxel/PRISM (find `GSE270828`)
- Authors' analysis code: https://github.com/NoonanLab/Noble_et_al_2024

**Do not upload the 2.31 GB GEO RDS here.** This notebook expects one processed `GSE270828*.h5ad` file saved in Google Drive or a direct individual Google Drive share URL. Do not use the *PRISM entire collection folder* URL (it contains many datasets).

## 1. Install minimal packages and connect Google Drive

Use a standard Colab runtime. This reads `.obs` metadata in backed mode and does not load the full expression matrix into RAM.

In [ ]:
%pip -q install "anndata>=0.10,<0.13" "pandas>=2.0,<3.0" h5py gdown
from google.colab import drive
drive.mount('/content/drive')

## 2. Tell Colab where your *individual* `.h5ad` file is

**Option A (preferred):** open PRISM's shared Google Drive folder, select only `GSE270828`'s `.h5ad` file, choose **Add shortcut to Drive** or copy it into your own Drive, then put its real path below. A shortcut may need to be copied first if the file is not readable in Colab.

**Option B:** paste the individual file's share link in `INDIVIDUAL_FILE_URL`, which can be downloaded with `gdown` if Google Drive allows it.

If you have an archive `.tar.gz`, extract only the GSE270828 `.h5ad` file in Drive before proceeding. Do not download the whole PRISM folder.

In [ ]:
# @title Input location (edit only one of these)
H5AD_DRIVE_PATH = "/content/drive/MyDrive/GSE270828_merged.h5ad"  # @param {type:"string"}
INDIVIDUAL_FILE_URL = ""  # @param {type:"string"}

import os, pathlib, json
from pathlib import Path
import gdown

path=Path(H5AD_DRIVE_PATH)
if not path.is_file() and INDIVIDUAL_FILE_URL.strip():
    if 'folders/' in INDIVIDUAL_FILE_URL:
        raise ValueError('This is a folder URL, not an individual H5AD file URL. Select the GSE270828 file only.')
    path=Path('/content/GSE270828_merged.h5ad')
    if not path.is_file():
        gdown.download(url=INDIVIDUAL_FILE_URL, output=str(path), quiet=False, fuzzy=True)
if not path.is_file():
    raise FileNotFoundError(
      'H5AD file not found. Download/select only GSE270828 processed H5AD from the PRISM Google Drive folder, '
      'save it as MyDrive/GSE270828_merged.h5ad, and rerun this cell. '
      'Do not paste the whole PRISM folder link.'
    )
if not str(path).lower().endswith('.h5ad'):
    raise ValueError('Expected an .h5ad file. Extract archives first, then point to the .h5ad.')
print('Input:',path,'size (GB):',round(path.stat().st_size/1e9,3))

## 3. Run predefined QC code (no biological inference)

This script audits all available metadata columns, counts by screened HAR, guide assignment, replicate labels and lane labels, and labels possible controls **heuristically**. It does **not** treat a guide or sequencing lane as an independent donor.

In [ ]:
#!/usr/bin/env python3
"""Metadata-only QC of GSE270828-derived AnnData (no expression-effect inference).

Usage: python stage7_metadata_qc.py input.h5ad output_dir --screen-labels screen_labels.txt
Requires: anndata, pandas, numpy. Reads metadata with backed='r' to limit memory.
"""
import argparse
import json
import re
import sys
from pathlib import Path
import pandas as pd

CONTROL_RE = re.compile(r'(?:^ntc\b|non[ -]?target|negative[ -]?control|scrambl|^ctrl\b|^control\b)', re.I)


def clean_text(s):
    return s.astype('string').fillna('<missing>').str.strip().replace({'': '<missing>'})


def aggregate_obs(obs, screen_labels=None):
    """Return *descriptive* summary dataframes and JSON-serializable summary."""
    obs = obs.copy()
    if 'har' not in obs.columns:
        raise ValueError('Required column `har` absent. Inspect original obs columns; no guessing.')
    for k in ('har', 'guide', 'replicate', 'lane', 'perturbation_name'):
        if k in obs.columns:
            obs[k] = clean_text(obs[k])
    obs['har'] = clean_text(obs['har'])

    n = len(obs)
    har_counts = obs.groupby('har', dropna=False, observed=True).size().reset_index(name='n_cells').sort_values('n_cells', ascending=False)
    har_counts['control_like_label_only'] = har_counts['har'].astype(str).map(lambda x: bool(CONTROL_RE.search(x)))
    controls = har_counts[har_counts['control_like_label_only']].copy()
    output = {'har_cell_counts.tsv': har_counts, 'possible_control_labels.tsv': controls}

    for col, filename in [('replicate','har_by_replicate_counts.tsv'), ('guide','har_by_guide_counts.tsv'), ('lane','har_by_lane_counts.tsv')]:
        if col in obs.columns:
            tab = obs.groupby(['har', col], observed=True, dropna=False).size().reset_index(name='n_cells')
            output[filename] = tab.sort_values(['har',col])
    if 'replicate' in obs.columns and 'lane' in obs.columns:
        output['replicate_by_lane_counts.tsv'] = obs.groupby(['replicate','lane'],observed=True).size().reset_index(name='n_cells')

    if screen_labels is not None:
        base=pd.DataFrame({'screened_HAR': sorted(set(map(str,screen_labels)))})
        c=har_counts[['har','n_cells']].rename(columns={'har':'screened_HAR','n_cells':'cells_in_this_processed_subset'})
        joined=base.merge(c,on='screened_HAR',how='left')
        joined['cells_in_this_processed_subset']=joined['cells_in_this_processed_subset'].fillna(0).astype('int64')
        joined['present_in_this_processed_subset']=joined['cells_in_this_processed_subset']>0
        if 'replicate' in obs.columns:
            rc=obs.groupby('har',observed=True)['replicate'].nunique().rename('n_observed_replicate_labels')
            joined=joined.join(rc, on='screened_HAR')
            joined['n_observed_replicate_labels']=joined['n_observed_replicate_labels'].fillna(0).astype('int64')
        if 'guide' in obs.columns:
            gc=obs.groupby('har',observed=True)['guide'].nunique().rename('n_distinct_assigned_guides')
            joined=joined.join(gc,on='screened_HAR')
            joined['n_distinct_assigned_guides']=joined['n_distinct_assigned_guides'].fillna(0).astype('int64')
        output['screen_180_HAR_coverage.tsv']=joined
    else: joined=None

    meta=[]
    for col in obs.columns:
        ser=obs[col]
        meta.append({'column_name':col,'dtype':str(ser.dtype),'n_missing':int(ser.isna().sum()) if col not in ['har','guide','replicate','lane','perturbation_name'] else int((ser=='<missing>').sum()), 'unique_nonmissing_values':int(ser.nunique(dropna=True))})
    output['obs_metadata_schema.tsv']=pd.DataFrame(meta)
    summary={
      'source_type':'PRISM processed subset if using the linked PRISM AnnData; confirm provenance',
      'n_cells_in_uploaded_file':int(n),
      'n_distinct_har_labels_in_uploaded_file':int(har_counts.shape[0]),
      'n_control_like_har_labels_heuristic_not_confirmed':int(controls.shape[0]),
      'replicate_column_present':bool('replicate' in obs.columns),
      'n_distinct_replicate_labels':int(obs['replicate'].nunique()) if 'replicate' in obs.columns else None,
      'n_distinct_lane_labels':int(obs['lane'].nunique()) if 'lane' in obs.columns else None,
      'guide_column_present':bool('guide' in obs.columns),
      'n_distinct_guide_labels':int(obs['guide'].nunique()) if 'guide' in obs.columns else None,
      'n_screened_HAR_in_original_design':int(len(set(map(str,screen_labels)))) if screen_labels is not None else None,
      'n_screened_HAR_detected_in_this_processed_subset':int(joined['present_in_this_processed_subset'].sum()) if joined is not None else None,
      'independent_donors_inferred':False,
      'expression_effect_statistics_computed':False,
      'original_188k_cells_recovered':False,
      'negative_controls_confirmed':False,
      'interpretation':'Descriptive metadata QC only. Check controls/replicates/count semantics and coverage before differential expression.'
    }
    return output, summary


def audit_h5ad(path, output_dir, screen_labels=None):
    try:
        import anndata as ad
    except ImportError as e:
        raise SystemExit('Install anndata: pip install anndata h5py pandas numpy') from e
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    a=ad.read_h5ad(path,backed='r')
    try:
        obs=a.obs.copy()
        tabs, summary=aggregate_obs(obs,screen_labels)
        summary.update({
          'input_file':str(path),
          'n_features_in_h5ad':int(a.n_vars),
          'n_obs_in_h5ad':int(a.n_obs),
          'available_obsm':list(a.obsm.keys()),
          'has_raw_attribute':bool(a.raw is not None),
          'source_file_size_bytes':Path(path).stat().st_size
        })
        for name,tab in tabs.items():tab.to_csv(out/name,sep='\t',index=False)
        (out/'metadata_qc_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
        (out/'README.txt').write_text('Metadata-only diagnostic. No gene expression effects, P-values, validation or causal claims computed.\\nOriginal source: GSE270828; alternative processed PRISM subset must be checked for processing and attrition.\\n',encoding='utf-8')
        return summary
    finally:
        if getattr(a, 'file',None) is not None: a.file.close()


def main():
    p=argparse.ArgumentParser();p.add_argument('h5ad');p.add_argument('output_dir');p.add_argument('--screen-labels')
    a=p.parse_args()
    screen=None
    if a.screen_labels:
        screen=[x.strip() for x in Path(a.screen_labels).read_text(encoding='utf-8').splitlines() if x.strip()]
    print(json.dumps(audit_h5ad(a.h5ad,a.output_dir,screen),indent=2))

In [ ]:
# Verified 180 HAR labels from prior guide-file audit.
SCREENED_HAR_LABELS = ['2xHAR.116', '2xHAR.128', '2xHAR.13', '2xHAR.132', '2xHAR.135', '2xHAR.165', '2xHAR.17', '2xHAR.177', '2xHAR.180', '2xHAR.183', '2xHAR.186', '2xHAR.20', '2xHAR.213', '2xHAR.215', '2xHAR.223', '2xHAR.234', '2xHAR.235', '2xHAR.238', '2xHAR.24', '2xHAR.240', '2xHAR.256', '2xHAR.258', '2xHAR.274', '2xHAR.287', '2xHAR.3', '2xHAR.30', '2xHAR.305', '2xHAR.310', '2xHAR.313', '2xHAR.33', '2xHAR.332', '2xHAR.333', '2xHAR.341', '2xHAR.346', '2xHAR.352', '2xHAR.377', '2xHAR.385', '2xHAR.39', '2xHAR.413', '2xHAR.419', '2xHAR.435', '2xHAR.437', '2xHAR.446', '2xHAR.471', '2xHAR.487', '2xHAR.49', '2xHAR.491', '2xHAR.492', '2xHAR.509', '2xHAR.51', '2xHAR.510', '2xHAR.531', '2xHAR.536', '2xHAR.62', '2xHAR.65', '2xHAR.67', '2xHAR.70', '2xHAR.82', '2xHAR.87', '2xHAR.99', 'HACNS10', 'HACNS105', 'HACNS125', 'HACNS129', 'HACNS149', 'HACNS15', 'HACNS155', 'HACNS174', 'HACNS188', 'HACNS196', 'HACNS198', 'HACNS205', 'HACNS238', 'HACNS239', 'HACNS255', 'HACNS259', 'HACNS278', 'HACNS295', 'HACNS325', 'HACNS33', 'HACNS332', 'HACNS336', 'HACNS368', 'HACNS377', 'HACNS38', 'HACNS383', 'HACNS420', 'HACNS423', 'HACNS426', 'HACNS439', 'HACNS442', 'HACNS454', 'HACNS457', 'HACNS464', 'HACNS480', 'HACNS497', 'HACNS50', 'HACNS52', 'HACNS522', 'HACNS544', 'HACNS549', 'HACNS551', 'HACNS563', 'HACNS567', 'HACNS579', 'HACNS583', 'HACNS584', 'HACNS587', 'HACNS588', 'HACNS607', 'HACNS635', 'HACNS636', 'HACNS643', 'HACNS656', 'HACNS659', 'HACNS67', 'HACNS671', 'HACNS674', 'HACNS682', 'HACNS684', 'HACNS686', 'HACNS694', 'HACNS698', 'HACNS703', 'HACNS71', 'HACNS712', 'HACNS719', 'HACNS722', 'HACNS734', 'HACNS735', 'HACNS754', 'HACNS76', 'HACNS761', 'HACNS762', 'HACNS763', 'HACNS772', 'HACNS780', 'HACNS786', 'HACNS798', 'HACNS805', 'HACNS839', 'HACNS851', 'HACNS858', 'HACNS860', 'HACNS865', 'HACNS870', 'HACNS880', 'HACNS901', 'HACNS909', 'HACNS915', 'HACNS924', 'HACNS925', 'HACNS927', 'HACNS938', 'HACNS96', 'HACNS962', 'HACNS964', 'HACNS968', 'HACNS975', 'HACNS976', 'HACNS99', 'HAR106', 'HAR108', 'HAR116', 'HAR149', 'HAR154', 'HAR156', 'HAR160', 'HAR165', 'HAR169', 'HAR17', 'HAR172', 'HAR181', 'HAR184', 'HAR190', 'HAR198', 'HAR202', 'HAR25', 'HAR67', 'HAR93']

OUT=Path('/content/HAR_Stage7_Metadata_Outputs')
summary=audit_h5ad(path, OUT, SCREENED_HAR_LABELS)
print(json.dumps(summary,indent=2))
print('Files:',*[p.name for p in OUT.iterdir()],sep='\n - ')

## 4. Export only small QC tables (do NOT upload large expression data)

The downloaded ZIP contains metadata summaries, not individual cell counts or confidential records. Upload it into the ChatGPT conversation.

In [ ]:
import shutil
from google.colab import files
zip_path=shutil.make_archive('/content/HAR_Stage7_Metadata_QC_Outputs','zip',OUT)
print('Output ZIP:',zip_path, 'bytes:',Path(zip_path).stat().st_size)
files.download(zip_path)